# **KNN Imputer for Missing Data**

### Topic Roadmap

**1. Dataset Loading & Exploration**

**2. Train-Test Split**

**3. KNN Imputation (Distance-Based)**
- 3.1 Implementation
- 3.2 Model Evaluation

**4. Baseline Comparison (Simple Imputation)**
- 4.1 Implementation
- 4.2 Model Evaluation
**5. Key Revision Notes**

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

### **1. Dataset Loading & Exploration**

Load the core numerical and ordinal features from the Titanic dataset and identify the percentage of missing values.

In [9]:
# Load specific columns
cols = ['Age', 'Pclass', 'Fare', 'Survived']
df = pd.read_csv('docs/Lecture-026-train.csv', usecols=cols)

# Check missing value percentages
print("Percentage of missing values:\n", df.isnull().mean() * 100)

Percentage of missing values:
 Survived     0.00000
Pclass       0.00000
Age         19.86532
Fare         0.00000
dtype: float64


### **2. Train-Test Split**

Separate the target variable from features, followed by an 80/20 split. This step must precede imputation to avoid data leakage.

In [3]:
# Separate features and target
X = df.drop(columns=['Survived'])
y = df['Survived']

# Perform Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)

X_train.head()

,Pclass,Age,Fare
30,1,40.0,27.7208
10,3,4.0,16.7000
873,3,47.0,9.0000
182,3,9.0,31.3875
876,3,20.0,9.8458


### **3. KNN Imputation (Distance-Based)**

`KNNImputer` estimates missing values using the k-Nearest Neighbors approach. Each missing value is imputed using the mean value from the `n_neighbors` nearest rows found in the training set.

### **3.1 Implementation**

In [4]:
# Initialize KNNImputer using 3 neighbors and distance-based weighting
knn = KNNImputer(n_neighbors=3, weights='distance')

# Fit on training data and transform both train and test sets
X_train_knn = knn.fit_transform(X_train)
X_test_knn = knn.transform(X_test)

### **3.2 Model Evaluation**

Train a Logistic Regression model on the KNN-imputed data to benchmark performance.

In [5]:
lr_knn = LogisticRegression()
lr_knn.fit(X_train_knn, y_train)

y_pred_knn = lr_knn.predict(X_test_knn)
print(f"Accuracy (KNN Imputer): {accuracy_score(y_test, y_pred_knn):.4f}")

Accuracy (KNN Imputer): 0.7039


### **4. Baseline Comparison (Simple Imputation)**

To validate the effectiveness of the KNN approach, compare it against a standard mean imputation baseline.

### **4.1 Implementation**

In [6]:
# Initialize standard mean imputer
si = SimpleImputer(strategy='mean')

# Fit and transform
X_train_si = si.fit_transform(X_train)
X_test_si = si.transform(X_test)

### **4.2 Model Evaluation**

In [7]:
lr_si = LogisticRegression()
lr_si.fit(X_train_si, y_train)

y_pred_si = lr_si.predict(X_test_si)
print(f"Accuracy (Simple Imputer): {accuracy_score(y_test, y_pred_si):.4f}")

Accuracy (Simple Imputer): 0.6927


### **Key Revision Notes**

- **KNN Imputer Mechanics:** Finds the $k$ nearest rows (neighbors) that have non-missing values for the target feature and imputes the missing spot using their mean.
- **`weights='distance'`:** Closer neighbors have a stronger influence on the imputed value than neighbors further away. The default is `'uniform'`.
- **Performance Trade-offs:** KNN Imputation is usually more accurate than Simple Imputation (as seen in the accuracy jump from ~0.69 to ~0.71) because it preserves multivariate relationships. However, it is computationally more expensive during both training and inference.
- **Scaling Requirement:** Since KNN relies on distance metrics (Euclidean distance), you should ideally scale your features (e.g., using `StandardScaler` or `MinMaxScaler`) before applying the imputer if the features are on vastly different scales.